# Migrate existing recordings into numbered subdirectories

This notebook converts the existing flat raw-recording layout:

```text
FLIC_<subject>/<activity>/GKA/<files>
FLIC_<subject>/<activity>/Neon/<files>
```

into:

```text
FLIC_<subject>/<activity>/GKA/1/<files>
FLIC_<subject>/<activity>/Neon/1/<files>
```

It only moves entries stored directly inside `GKA`, `Neon`, or `NEON`. Existing numeric recording directories are left unchanged. The migration defaults to a dry run. Review the plan, then set `DRY_RUN = False` and rerun the final cell.


In [ ]:
from pathlib import Path
import os
import re
import shutil

# Change this to the root containing FLIC_<subject> directories.
RAW_DATASET_DIR = Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026")

# Keep this True for the first run. Set it to False only after reviewing the plan.
DRY_RUN = True

# The current pipeline uses `Neon`; `NEON` is included for older datasets.
RECORDING_CONTAINER_NAMES = ("GKA", "Neon", "NEON")
MIGRATED_RECORDING_NUMBER = 1


In [ ]:
def build_migration_plan(dataset_root: Path):
    """Find flat recording entries that should move into recording 1."""
    dataset_root = dataset_root.expanduser().resolve()
    if not dataset_root.is_dir():
        raise NotADirectoryError(f"Dataset root is not a directory: {dataset_root}")

    migration_groups = []
    conflicts = []
    seen_container_ids = set()

    subject_dirs = sorted(
        (path for path in dataset_root.iterdir() if path.is_dir() and re.fullmatch(r"FLIC_\d+", path.name)),
        key=lambda path: path.name,
    )
    for subject_dir in subject_dirs:
        for activity_dir in sorted((path for path in subject_dir.iterdir() if path.is_dir()), key=lambda path: path.name):
            for container_name in RECORDING_CONTAINER_NAMES:
                container_dir = activity_dir / container_name
                if not container_dir.is_dir():
                    continue

                # On case-insensitive filesystems, `Neon` and `NEON` can
                # identify the same directory. Process each filesystem object once.
                container_stat = container_dir.stat()
                container_id = (container_stat.st_dev, container_stat.st_ino)
                if container_id in seen_container_ids:
                    continue
                seen_container_ids.add(container_id)

                # Numeric directories are already organized recording attempts.
                flat_entries = sorted(
                    (entry for entry in container_dir.iterdir() if not (entry.is_dir() and entry.name.isdigit())),
                    key=lambda path: path.name,
                )
                if not flat_entries:
                    continue

                target_dir = container_dir / str(MIGRATED_RECORDING_NUMBER)
                staging_dir = container_dir / f".recording-{MIGRATED_RECORDING_NUMBER}-migration-staging"
                if target_dir.exists():
                    conflicts.append(f"Cannot move flat entries because target already exists: {target_dir}")
                    continue
                if staging_dir.exists():
                    conflicts.append(f"A prior migration staging directory still exists: {staging_dir}")
                    continue

                migration_groups.append((container_dir, target_dir, staging_dir, flat_entries))

    return migration_groups, conflicts


def print_migration_plan(migration_groups, conflicts):
    move_count = sum(len(entries) for _, _, _, entries in migration_groups)
    print(f"Planned container migrations: {len(migration_groups)}")
    print(f"Planned entries to move: {move_count}")
    for container_dir, target_dir, _, entries in migration_groups:
        print(f"\n{container_dir} -> {target_dir}")
        for entry in entries:
            print(f"  {entry.name}")
    if conflicts:
        print("\nCONFLICTS (these containers will not be changed):")
        for conflict in conflicts:
            print(f"  {conflict}")


migration_groups, conflicts = build_migration_plan(RAW_DATASET_DIR)
print_migration_plan(migration_groups, conflicts)


In [ ]:
def migrate_recording_groups(migration_groups, dry_run: bool = True):
    """Move each flat container into /1 with per-container rollback."""
    if dry_run:
        print("DRY RUN: no files were moved. Set DRY_RUN = False to apply this plan.")
        return

    for container_dir, target_dir, staging_dir, planned_entries in migration_groups:
        # Recheck the plan immediately before changing this container.
        if target_dir.exists() or staging_dir.exists():
            raise FileExistsError(f"Target or staging directory appeared after planning: {container_dir}")
        for source_entry in planned_entries:
            if not source_entry.exists():
                raise FileNotFoundError(f"Planned source entry disappeared: {source_entry}")

        staging_dir.mkdir()
        moved_entries = []
        try:
            for source_entry in planned_entries:
                staged_entry = staging_dir / source_entry.name
                shutil.move(str(source_entry), str(staged_entry))
                moved_entries.append(staged_entry)

            # Rename the complete staging directory only after every entry moved.
            os.replace(staging_dir, target_dir)
            print(f"Migrated: {container_dir} -> {target_dir}")
        except Exception:
            # Restore entries already moved if this container fails partway through.
            for staged_entry in reversed(moved_entries):
                if staged_entry.exists():
                    shutil.move(str(staged_entry), str(container_dir / staged_entry.name))
            if staging_dir.exists():
                staging_dir.rmdir()
            raise


# Rebuild immediately before execution so a stale preview is never applied.
migration_groups, conflicts = build_migration_plan(RAW_DATASET_DIR)
if conflicts:
    print_migration_plan(migration_groups, conflicts)
    raise RuntimeError("Resolve the conflicts printed above before running the migration.")
migrate_recording_groups(migration_groups, dry_run=DRY_RUN)
